In [ ]:
# Context Window = Maximum tokens in a single request

Different models have different limits:

Gemini 2.5 Flash: ~1 million tokens
GPT-4: ~128K tokens
Claude 4: ~200K tokens

In [1]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r'C:\Users\sampa\OneDrive\Desktop\LLM_prompt_analyzer\.env')
API_KEY=os.getenv("GEMINI_API_KEY")

client=genai.Client(api_key=API_KEY)

In [ ]:
# Context Window in Conversations
In chat applications, the context window includes:

All previous messages (entire conversation history)
Current message
Response



As conversations grow, tokens accumulate!

In [2]:
messages = []

def chat(user_message):
    """Send message and track tokens"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )
    
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=messages
    )
    
    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )
    
    # Show token usage
    total_tokens = response.usage_metadata.total_token_count
    print(f"🤖: {response.text}")
    print(f"📊 Total tokens used: {total_tokens} (includes ALL {len(messages)} messages)\n")
    
    return response.text

# Start conversation
print("👤: Hi! My name is Yash")
chat("Hi! My name is Yash")

print("👤: I'm 25 years old")
chat("I'm 25 years old")

print("👤: I love Python programming")
chat("I love Python programming")

print("👤: I work as a software engineer")
chat("I work as a software engineer")

print("👤: What's my name?")
chat("What's my name?")

👤: Hi! My name is Yash
🤖: Hi Yash! It's nice to meet you. How can I help you today?
📊 Total tokens used: 25 (includes ALL 2 messages)

👤: I'm 25 years old
🤖: Thanks for letting me know, Yash! That's a great age. Is there anything specific you'd like to talk about or ask me, perhaps related to your age, career, or anything else?
📊 Total tokens used: 77 (includes ALL 4 messages)

👤: I love Python programming
🤖: That's fantastic, Yash! Python is an incredibly popular and versatile language. I can see why you'd love it.

What do you enjoy most about Python? Are you working on any particular projects, or do you have any specific areas of Python you're most interested in, like:

*   **Web development** (e.g., Django, Flask)
*   **Data science and machine learning** (e.g., Pandas, NumPy, Scikit-learn, TensorFlow)
*   **Automation and scripting**
*   **Game development**
*   **General programming and problem-solving**

I'd love to hear more about your passion for Python!
📊 Total tokens used: 2

'Your name is **Yash**.'

In [3]:
MAX_MESSAGES = 6  # Keep only last 6 messages (3 exchanges)

def chat_with_limit(user_message):
    """Chat with context window management"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )
    
    # Remove old messages if too many
    if len(messages) > MAX_MESSAGES:
        removed = messages.pop(0)  # Remove oldest
        print(f"🗑️  Removed old message: {removed.parts[0].text[:50]}...\n")
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages
    )
    
    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )
    
    print(f"🤖: {response.text}")
    print(f"📊 Messages in memory: {len(messages)}\n")
    
    return response.text

# Reset and try again
messages = []

print("👤: My favorite color is blue")
chat_with_limit("My favorite color is blue")

print("👤: I have a dog named Max")
chat_with_limit("I have a dog named Max")

print("👤: I live in Mumbai")
chat_with_limit("I live in Mumbai")

print("👤: I enjoy hiking")
chat_with_limit("I enjoy hiking")

# This will forget the first message!
print("👤: What's my favorite color?")
chat_with_limit("What's my favorite color?")

👤: My favorite color is blue
🤖: That's a lovely choice! Blue is a fantastic color.

It's so versatile and can evoke so many different feelings and images – from calming oceans and clear skies to vibrant electric blues and sophisticated navies.

What do you like most about blue? Or do you have a favorite shade?
📊 Messages in memory: 2

👤: I have a dog named Max
🤖: Aw, Max! That's such a classic and lovely dog name.

What kind of dog is Max? And what's he like? I bet he brings a lot of joy to your life!
📊 Messages in memory: 4

👤: I live in Mumbai
🤖: Mumbai! What a fantastic city.

It's known for its incredible energy, vibrant culture, amazing food, and bustling life. I bet living there is quite an experience!
📊 Messages in memory: 6

👤: I enjoy hiking
🗑️  Removed old message: My favorite color is blue...

🤖: That's fantastic! Mumbai is perfectly situated for some great hiking opportunities, especially once you get out of the immediate city bustle.

The **Western Ghats** are practically 

"Haha! That's a great question, and one I definitely don't know!\n\nOur conversation has covered your dog, your city, and your love for hiking, but your favorite color hasn't come up yet.\n\nWhat *is* your favorite color? I'm curious now!"